In [2]:
from datasets import load_dataset

dtaset = load_dataset("cis-lmu/Glot500", "nbl_Latn", split = "train", streaming = True)

print("--- Raw Data Sample ---")
for i, row in enumerate(dtaset):
    if i >= 5:
        break
    
    # The raw scraped data is stored under the 'text' key
    # Truncating the output here for readability in the console
    print(f"Row {i+1}: {row['text'][:150]}...")

    output_filename = 'raw_data.csv'
    dtaset.to_csv(output_filename, index=False)

print(f"Successfully created {output_filename}")

/Users/Valerie/Desktop/MAP4905 2/DataPreprocessing/valsVenv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--- Raw Data Sample ---
Row 1: Ukudla kwendabuko okufana nomrorho, imihlaza, i-tihove, i-semphemphe, i-potele, amathumbu nomrhoru neenselo ezifana nerhemere, akusimatshwayo kuphela ...


Creating CSV from Arrow format: 100%|██████████| 21/21 [00:05<00:00,  3.98ba/s]


Row 2: Kghani kunento okumele uyenze Iye, eqinisweni kunjalo: Elinye nelinye ilungelo olibizako, obunye nobunye ubunikazi buza nomsebenzi/nesibopho sakhona! ...


Creating CSV from Arrow format: 100%|██████████| 21/21 [00:03<00:00,  5.88ba/s]


...

Row 4: Imihlangano Yekomidi Yewodi - le mihlangano evamileko yamalungu wekomidi yewodi. Imihlangano le kufuze bona inande ibanjwa, kobanyana amalungu wekomid...


Creating CSV from Arrow format: 100%|██████████| 21/21 [00:03<00:00,  5.33ba/s]


Row 5: (1) Omunye nomunye odosa iinyanga ezidlula eziyi 12 ejele ngaphandle kwehlawulo ngaphakathi kweRiphabhligi lokha umThethosisekelo omutjha uthoma ukuse...


Creating CSV from Arrow format: 100%|██████████| 21/21 [00:04<00:00,  5.09ba/s]


Successfully created raw_data.csv


### Feature Extraction: Extract numerical features including token-to-character ratios and n-gram frequencies.

In [ ]:
# Extracting numerical features from the raw text data
import numpy as np

sampleSize = 10000
tokenLneghts = []
charTokenRations = []


for i, row in enumerate(dtaset):
    if i >= sampleSize:
        break
        
    text = row['text']
    char_length = len(text)
    
    # A basic token count approximation (splitting by whitespace)
    tokens = text.split()
    token_count = len(tokens)
    
    # Filter out empty strings and artifacts to avoid division by zero
    if token_count > 0:
        ratio = char_length / token_count
        
        tokenLneghts.append(token_count)
        charTokenRations.append(ratio)

# Convert to NumPy arrays for efficient mathematical modeling
lengths_array = np.array(tokenLneghts)
ratios_array = np.array(charTokenRations)

print("\n--- Baseline Statistical Profile ---")
print(f"Mean Token Length: {np.mean(lengths_array):.2f}")
print(f"Variance in Length: {np.var(lengths_array):.2f}")
print(f"Standard Deviation: {np.std(lengths_array):.2f}")
print(f"Mean Char-to-Token Ratio: {np.mean(ratios_array):.2f}")


In [ ]:
import matplotlib.pyplot as plt

# 1. Set up the figure size for a clean, readable plot
plt.figure(figsize=(10, 6))

# 2. Plot the histogram for the Character-to-Token Ratios
# We use 50 bins to get a detailed look at the distribution's shape
plt.hist(ratios_array, bins=50, color='skyblue', edgecolor='black', alpha=0.7)

# 3. Add titles and labels
plt.title('Raw Distribution of Character-to-Token Ratios', fontsize=14, fontweight='bold')
plt.xlabel('Character-to-Token Ratio', fontsize=12)
plt.ylabel('Frequency (Number of Sentences)', fontsize=12)

# 4. Add a vertical line to show where the Mean falls
mean_ratio = np.mean(ratios_array)
plt.axvline(mean_ratio, color='red', linestyle='dashed', linewidth=2, label=f'Mean: {mean_ratio:.2f}')

# 5. Add a grid and legend for readability
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend()

# 6. Display the plot
plt.tight_layout()
plt.show()

### Using Beta distribution to better understand the usability of this corpora data

In [ ]:
from scipy.stats import beta
import matplotlib.pyplot as plt
# 1. Min-Max Scaling to compress data into (0, 1) bounds
min_val = np.min(ratios_array)
max_val = np.max(ratios_array)

# Add a tiny epsilon so we don't get exactly 0.0 or 1.0 (which breaks MLE)
epsilon = 1e-5
normalized_ratios = (ratios_array - min_val) / (max_val - min_val)
normalized_ratios = np.clip(normalized_ratios, epsilon, 1 - epsilon)

# 2. Fit the Beta distribution to the normalized data
a, b, loc, scale = beta.fit(normalized_ratios, floc=0, fscale=1)

# 3. Plotting
x = np.linspace(beta.ppf(0.01, a, b), beta.ppf(0.99, a, b), 100)

fig, ax = plt.subplots(1, 1)
ax.plot(x, beta.pdf(x, a, b), 'r-', lw=5, alpha=0.6, label='Beta PDF')
# Make sure to plot the normalized data, not the raw data!
ax.hist(normalized_ratios, bins=30, density=True, alpha=0.6, color='g', label='Data')
ax.set_title('Fitted Beta Distribution to Normalized Char-to-Token Ratios')
ax.set_xlabel('Normalized Char-to-Token Ratio (0 to 1)')
ax.set_ylabel('Density')
ax.legend()
plt.show()

In [ ]:
#Gamma Distribution
from scipy.stats import gamma

# Fit Gamma directly to the raw, un-scaled ratios
# floc=0 keeps the left bound at 0
fit_alpha, fit_loc, fit_scale = gamma.fit(ratios_array, floc=0)

x = np.linspace(gamma.ppf(0.01, fit_alpha, loc=fit_loc, scale=fit_scale), 
                gamma.ppf(0.99, fit_alpha, loc=fit_loc, scale=fit_scale), 100)

fig, ax = plt.subplots(1, 1)
ax.plot(x, gamma.pdf(x, fit_alpha, loc=fit_loc, scale=fit_scale), 'r-', lw=5, alpha=0.6, label='Gamma PDF')
ax.hist(ratios_array, bins=30, density=True, alpha=0.6, color='g', label='Raw Data')
ax.set_title('Fitted Gamma Distribution to Char-to-Token Ratios')
ax.set_xlabel('Raw Char-to-Token Ratio')
ax.set_ylabel('Density')
ax.legend()
plt.show()

As can be seen, the beta distribution produced data that looks normal/ gaussian. To the contrary, the gamma distribution (which was not normalized unlike the beta distribution) is slightly right skewed. This indicates that a CDF approach should be used to clean the data instead of standard deviations in order to keep the natural flow of character lenghts of words. 

In [ ]:
# Implementing a 3-sigma filter

# 1. Calculate mu and sigma from the baseline 'ratios_array' you built earlier
mu = np.mean(ratios_array)
sigma = np.std(ratios_array)

# 2. Define the mathematical bounds (the 99.7% confidence interval)
lower_bound = mu - (3 * sigma)
upper_bound = mu + (3 * sigma)

print(f"Filter active. Mean (μ): {mu:.2f}, Std Dev (σ): {sigma:.2f}")
print(f"Acceptable Ratio Bounds: [{lower_bound:.2f}, {upper_bound:.2f}]\n")


# 4. Define the mathematical indicator function
def filter_by_zscore(example):
    text = example['text']
    char_length = len(text)
    tokens = text.split()
    token_count = len(tokens)
    
    # Drop empty strings to avoid division by zero errors
    if token_count == 0:
        return False
        
    ratio = char_length / token_count
    
    # Return True if the ratio falls within our 3-sigma bounds
    return lower_bound <= ratio <= upper_bound

# 5. Apply the filter to the stream
# This creates a new streaming dataset that ONLY yields statistically valid rows
clean_dataset = dtaset.filter(filter_by_zscore)

# 6. Test the pipeline by pulling the first 5 mathematically verified rows
print("--- Verified Clean Data ---")
# .take(5) safely pulls just the first 5 rows that pass the filter
for i, row in enumerate(clean_dataset.take(5)):
    print(f"Clean Row {i+1}: {row['text'][:150]}...")

In [ ]:
import csv
# 1. Establish the Min-Max scaling factors from your Phase 1 sample
# (Assuming 'ratios_array' is your original 10,000-row sample)
sample_min = np.min(ratios_array)
sample_max = np.max(ratios_array)

# Normalize the sample to fit the Beta distribution (adding epsilon to prevent 0 or 1)
epsilon = 1e-5
normalized_sample = (ratios_array - sample_min) / (sample_max - sample_min)
normalized_sample = np.clip(normalized_sample, epsilon, 1 - epsilon)

# 2. Fit the Beta distribution to the normalized sample
a, b, loc, scale = beta.fit(normalized_sample, floc=0, fscale=1)

# 3. Calculate the Beta bounds (1st and 99th percentiles in the normalized space)
beta_lower = beta.ppf(0.01, a, b)
beta_upper = beta.ppf(0.99, a, b)

print("--- Beta Distribution Filter Active ---")
print(f"Sample Min: {sample_min:.2f}, Sample Max: {sample_max:.2f}")
print(f"Normalized Acceptable Bounds: [{beta_lower:.4f}, {beta_upper:.4f}]\n")

# Filtering done by Beta and saved as csv
def filter_by_beta(example):
    text = example['text']
    char_length = len(text)
    tokens = text.split()
    token_count = len(tokens)
    
    if token_count == 0:
        return False
        
    raw_ratio = char_length / token_count
    
    # Normalize the incoming ratio using the SAMPLE's min and max
    normalized_ratio = (raw_ratio - sample_min) / (sample_max - sample_min)
    
    # Return True ONLY if the normalized ratio is within the Beta bounds
    return beta_lower <= normalized_ratio <= beta_upper

# 6. Apply the filter to create the clean stream
clean_beta_dataset = dtaset.filter(filter_by_beta)

# 7. Stream directly to CSV
def save_stream_to_csv(streamed_dataset, output_filename="nbl_beta_filtered.csv", max_rows=50000):
    print(f"Writing clean, Beta-filtered data to '{output_filename}'...")
    
    with open(output_filename, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(['text'])
        
        rows_saved = 0
        for row in streamed_dataset:
            writer.writerow([row['text']])
            rows_saved += 1
            
            if rows_saved % 5000 == 0:
                print(f"Status: {rows_saved} valid rows saved...")
                
            if rows_saved >= max_rows:
                print(f"Reached target size of {max_rows} rows. Halting stream.")
                break
                
    print(f"\nPipeline complete! Successfully saved {rows_saved} rows.")

# Execute the pipeline
save_stream_to_csv(clean_beta_dataset, output_filename="nbl_beta_filtered.csv", max_rows=50000)

In [ ]:
import csv

def save_stream_to_csv(streamed_dataset, output_filename="cleaned_corpus.csv", max_rows=100000):
    """
    Progressively writes valid rows from a Hugging Face stream to a CSV file.
    """
    print(f"Starting pipeline. Writing clean data to '{output_filename}'...")
    
    # encoding='utf-8' is strictly required to prevent character corruption
    with open(output_filename, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        
        # Write the column header
        writer.writerow(['text'])
        
        rows_saved = 0
        
        # Iterate through the generator one row at a time
        for row in streamed_dataset:
            # Write the clean text to the CSV
            writer.writerow([row['text']])
            rows_saved += 1
            
            # Print a status update every 5,000 rows so you know it is working
            if rows_saved % 5000 == 0:
                print(f"Status: {rows_saved} clean rows saved...")
                
            # A safety valve to stop the stream once you have enough data
            if rows_saved >= max_rows:
                print(f"Reached target size of {max_rows} rows. Halting stream.")
                break
                
    print(f"\nPipeline complete! Successfully saved {rows_saved} rows.")

# --- How to execute the function ---
# Pass in the 'clean_dataset' variable you created in the previous step
save_stream_to_csv(clean_dataset, output_filename="nbl_clean_3sigma.csv", max_rows=50000)

In [ ]:
lower_bound = gamma.ppf(0.01, fit_alpha, loc=fit_loc, scale=fit_scale)
upper_bound = gamma.ppf(0.99, fit_alpha, loc=fit_loc, scale=fit_scale)

print("--- Gamma Distribution Filter Active ---")
print(f"1st Percentile (Lower Bound Cutoff): {lower_bound:.2f}")
print(f"99th Percentile (Upper Bound Cutoff): {upper_bound:.2f}\n")



def filter_by_gamma(example):
    text = example['text']
    char_length = len(text)
    tokens = text.split()
    token_count = len(tokens)
    
    # Drop empty strings to avoid division by zero errors
    if token_count == 0:
        return False
        
    ratio = char_length / token_count
    
    # Return True ONLY if the ratio falls within our asymmetric Gamma bounds
    return lower_bound <= ratio <= upper_bound

clean_gamma_dataset = dtaset.filter(filter_by_gamma)


def save_stream_to_csv(streamed_dataset, output_filename="cleaned_gamma_corpus.csv", max_rows=100000):
    print(f"Writing clean, Gamma-filtered data to '{output_filename}'...")
    
    with open(output_filename, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(['text']) # Header
        
        rows_saved = 0
        for row in streamed_dataset:
            writer.writerow([row['text']])
            rows_saved += 1
            
            if rows_saved % 5000 == 0:
                print(f"Status: {rows_saved} valid rows saved...")
                
            if rows_saved >= max_rows:
                print(f"Reached target size of {max_rows} rows. Halting stream.")
                break
                
    print(f"\nPipeline complete! Successfully saved {rows_saved} rows.")

# Execute the pipeline
save_stream_to_csv(clean_gamma_dataset, output_filename="nbl_gamma_filtered.csv", max_rows=50000)

### Comparitive Analysis of Data Cleaning:

In [ ]:
from scipy.stats import skew
import pandas as pd
def calculate_ratios_from_csv(filepath):
    """Reads the saved CSV and recalculates the char-to-token ratios for analysis."""
    try:
        df = pd.read_csv(filepath)
        ratios = []
        for text in df['text'].dropna():
            char_length = len(text)
            token_count = len(text.split())
            if token_count > 0:
                ratios.append(char_length / token_count)
        return np.array(ratios)
    except FileNotFoundError:
        print(f"File not found: {filepath}")
        return np.array([])

def generate_summary_stats(name, data_array):
    """Calculates rigorous mathematical statistics for a given array."""
    if len(data_array) == 0:
        return {}
    
    return {
        "Dataset": name,
        "N (Rows)": len(data_array),
        "Mean (μ)": np.mean(data_array),
        "Variance (σ²)": np.var(data_array),
        "Std Dev (σ)": np.std(data_array),
        "Skewness": skew(data_array),
        "Min": np.min(data_array),
        "Max": np.max(data_array)
    }

# 1. Load your datasets (Replace with your actual raw array and CSV filenames)
# We use the initial raw_ratios_array you created during Phase 1 for the baseline
raw_data = ratios_array 
gamma_data = calculate_ratios_from_csv("/Users/Valerie/Desktop/MAP4905/DataPreprocessing/nbl_gamma_filtered.csv")
beta_data = calculate_ratios_from_csv("/Users/Valerie/Desktop/MAP4905/DataPreprocessing/nbl_beta_filtered.csv")
sigma3_data = calculate_ratios_from_csv("/Users/Valerie/Desktop/MAP4905/DataPreprocessing/nbl_clean_3sigma.csv")

# 2. Compute the statistics
stats_list = [
    generate_summary_stats("Raw Baseline", raw_data),
    generate_summary_stats("Gamma Filtered (Asymmetric)", gamma_data),
    generate_summary_stats("Beta Filtered (Bounded)", beta_data),
    generate_summary_stats("3-Sigma Filtered", sigma3_data)
    
]

# 3. Format and display as a DataFrame for a clean, report-ready table
results_df = pd.DataFrame([s for s in stats_list if s])

# Round the numerical columns to 4 decimal places for academic presentation
numerical_cols = ["Mean (μ)", "Variance (σ²)", "Std Dev (σ)", "Skewness", "Min", "Max"]
results_df[numerical_cols] = results_df[numerical_cols].round(4)

print("\n--- Summary Statistics Comparison ---")
print(results_df.to_string(index=False))

In [ ]:
results_df.head()

### Statistical Distance and Hypothesis Testing

In [ ]:
from collections import Counter

def build_expected_distribution(clean_csv_path, top_k=50):
    """
    Reads a clean corpus and calculates the expected probabilities 
    for the most common character bigrams.
    """
    df = pd.read_csv(clean_csv_path)
    # Take a sample of 5,000 clean sentences to build the baseline
    clean_texts = df['text'].dropna().head(5000).tolist()
    
    bigrams = []
    for text in clean_texts:
        chars = list(text.lower())
        # Generate character bigrams (e.g., 'hello' -> 'he', 'el', 'll', 'lo')
        bigrams.extend([f"{chars[i]}{chars[i+1]}" for i in range(len(chars)-1)])
        
    counts = Counter(bigrams)
    total_bigrams = sum(counts.values())
    
    # Keep the top K most frequent bigrams. 
    # Everything else gets grouped into "OTHER" to prevent zero-division in Chi-Squared.
    most_common = counts.most_common(top_k)
    
    expected_probs = {bigram: count / total_bigrams for bigram, count in most_common}
    expected_probs["OTHER"] = 1.0 - sum(expected_probs.values())
    
    return expected_probs

# Calculate your baseline probabilities
expected_probabilities = build_expected_distribution("nbl_gamma_filtered.csv")
print("Baseline established. Top 5 bigrams:", list(expected_probabilities.items())[:5])

In [ ]:
from scipy.stats import chisquare
def filter_by_chisquare(sentence, expected_probs, alpha=0.01):
    """
    Runs a Chi-Squared goodness-of-fit test on a single sentence.
    Returns True if the sentence matches the expected distribution (Keep).
    Returns False if it is statistically anomalous (Drop).
    """
    chars = list(sentence.lower())
    if len(chars) < 10:
        # Sentences that are too short break the Chi-Squared assumptions
        return False 
        
    # Extract observed bigrams for this specific sentence
    obs_bigrams = [f"{chars[i]}{chars[i+1]}" for i in range(len(chars)-1)]
    obs_counts = Counter(obs_bigrams)
    total_obs = len(obs_bigrams)
    
    O_observed = []
    E_expected = []
    other_obs_count = 0
    
    # 1. Align observed counts with the expected probability keys
    for bigram, exp_p in expected_probs.items():
        if bigram == "OTHER":
            continue
            
        # If the bigram exists in the sentence, get its count. Otherwise, 0.
        O_observed.append(obs_counts.get(bigram, 0))
        # Expected count = probability * total bigrams in this sentence
        E_expected.append(exp_p * total_obs)
        
    # 2. Tally up all the strange/rare bigrams into the "OTHER" bucket
    for bigram, count in obs_counts.items():
        if bigram not in expected_probs:
            other_obs_count += count
            
    O_observed.append(other_obs_count)
    E_expected.append(expected_probs["OTHER"] * total_obs)
    
    # Add a tiny smoothing factor (1e-5) to Expected counts to avoid dividing by zero
    E_expected = np.array(E_expected) + 1e-5
    O_observed = np.array(O_observed)
    
    # 3. Run the Chi-Squared Test
    # Suppress warnings if frequencies are low, which happens often in sentence-level text
    with np.errstate(divide='ignore', invalid='ignore'):
        stat, p_value = chisquare(f_obs=O_observed, f_exp=E_expected)
    
    # 4. Hypothesis Testing Check
    # If p_value is less than alpha, it's an anomaly. Return False to drop it.
    if pd.isna(p_value) or p_value < alpha:
        return False
        
    return True

In [ ]:
#Create a wrapper function so the filter only takes 'example' as an argument
def chisquare_wrapper(example):
    return filter_by_chisquare(example['text'], expected_probabilities)

# Apply it to a stream
chi_clean_dataset = dtaset.filter(chisquare_wrapper)

In [ ]:
def add_chisquare_stats(example):
    """
    Calculates the Chi-Squared statistic and p-value for a sentence,
    and attaches them as new columns to the dataset.
    """
    chars = list(example['text'].lower())
    
    # Handle edge cases (too short to test)
    if len(chars) < 10:
        example['chi_stat'] = np.nan
        example['p_value'] = np.nan
        example['passed_chi_test'] = False
        return example
        
    obs_bigrams = [f"{chars[i]}{chars[i+1]}" for i in range(len(chars)-1)]
    obs_counts = Counter(obs_bigrams)
    total_obs = len(obs_bigrams)
    
    O_observed = []
    E_expected = []
    other_obs_count = 0
    
    # Align observed counts with the expected probability keys
    for bigram, exp_p in expected_probabilities.items():
        if bigram == "OTHER":
            continue
        O_observed.append(obs_counts.get(bigram, 0))
        E_expected.append(exp_p * total_obs)
        
    # Group rare bigrams into OTHER
    for bigram, count in obs_counts.items():
        if bigram not in expected_probabilities:
            other_obs_count += count
            
    O_observed.append(other_obs_count)
    E_expected.append(expected_probabilities["OTHER"] * total_obs)
    
    # Convert to float arrays explicitly to handle the tiny decimals
    O_observed = np.array(O_observed, dtype=float)
    E_expected = np.array(E_expected, dtype=float)
    
    # 1. Add smoothing factor to avoid division by zero
    E_expected += 1e-8
    
    # 2. Scale the Expected array to match the Observed sum
    E_expected = E_expected * (np.sum(O_observed) / np.sum(E_expected))
    
    # 3. Force exact floating-point equality (The Fix)
    exact_difference = np.sum(O_observed) - np.sum(E_expected)
    E_expected[-1] += exact_difference
    
    # Calculate the exact statistics
    with np.errstate(divide='ignore', invalid='ignore'):
        stat, p_value = chisquare(f_obs=O_observed, f_exp=E_expected)
    
    # Attach the math to the Hugging Face dataset row
    example['chi_stat'] = stat
    example['p_value'] = p_value
    
    # Flag it as True if the p-value is greater than or equal to our 0.01 alpha threshold
    example['passed_chi_test'] = not (pd.isna(p_value) or p_value < 0.01)
    
    return example

In [ ]:
# 1. Map the mathematical function to the streaming dataset
# (This adds the 'chi_stat', 'p_value', and 'passed_chi_test' columns)
mapped_dataset = dtaset.map(add_chisquare_stats)

# 2. Filter the stream to ONLY keep the clean data (where passed_chi_test is True)
chi_clean_dataset = mapped_dataset.filter(lambda x: x['passed_chi_test'])

print("Extracting a 5,000-row sample to compute summary statistics...")

# 3. Pull a sample of the clean stream into memory
results = []
for i, row in enumerate(chi_clean_dataset):
    if i >= 5000:
        break
    
    results.append({
        'text_length': len(row['text']),
        'chi_stat': row['chi_stat'],
        'p_value': row['p_value']
    })

# Convert the extracted sample into a Pandas DataFrame
df_clean_stats = pd.DataFrame(results)
df_clean_stats.head()

In [ ]:
anomalies_dataset = mapped_dataset.filter(lambda x: not x['passed_chi_test'])

print("--- 5 Rejected Anomalies (H0 Rejected: p < 0.01) ---")

# 2. Extract and print the first 5 failures
for i, row in enumerate(anomalies_dataset.take(10)):
    print(f"\n[Anomaly {i+1}]")
    print(f"Chi-Stat (χ²): {row['chi_stat']:.2f}")
    # Using .6f to show just how microscopic these p-values get
    print(f"p-value:       {row['p_value']:.6f}") 
    
    # Print the first 250 characters of the text to see the structure
    text = row['text']
    display_text = text[:250] + "..." if len(text) > 250 else text
    print(f"Text: {display_text}")

In [ ]:
print("--- Starting Anomaly Counter ---")

total_rows_processed = 0
total_anomalies_found = 0

# Iterate through your mapped dataset
for row in mapped_dataset:
    total_rows_processed += 1
    
    # If the row failed the Chi-Squared test, count it as an anomaly
    if not row['passed_chi_test']:
        total_anomalies_found += 1
        
    # Print a progress update every 10,000 rows
    if total_rows_processed % 10000 == 0:
        print(f"Processed {total_rows_processed} rows... Anomalies found so far: {total_anomalies_found}")

# Final Results
clean_rows = total_rows_processed - total_anomalies_found
anomaly_rate = (total_anomalies_found / total_rows_processed) * 100

print("\n--- Final Dataset Statistics ---")
print(f"Total Rows Processed: {total_rows_processed}")
print(f"Total Clean Rows:     {clean_rows}")
print(f"Total Anomalies:      {total_anomalies_found}")
print(f"Anomaly Rejection Rate: {anomaly_rate:.2f}%")

In [ ]:
print("Extracting samples for final mathematical validation...")

# 1. Extract a sample of the RAW dataset (Before filtering, but with chi_stats calculated)
raw_results = []
for i, row in enumerate(mapped_dataset):
    if i >= 5000: 
        break
    raw_results.append({
        'chi_stat': row['chi_stat'],
        'text_length': len(row['text'])
    })
df_raw = pd.DataFrame(raw_results).dropna()

# 2. Extract a sample of the CLEAN dataset (After Chi-Squared filtering)
clean_results = []
for i, row in enumerate(chi_clean_dataset):
    if i >= 5000: 
        break
    clean_results.append({
        'chi_stat': row['chi_stat'],
        'text_length': len(row['text'])
    })
df_clean = pd.DataFrame(clean_results).dropna()

# 3. Generate Summary Statistics to prove decreased variance
def get_validation_stats(df, column, dataset_name):
    return {
        "Dataset": dataset_name,
        "Metric": column,
        "N (Rows)": len(df),
        "Mean (μ)": np.mean(df[column]),
        "Variance (σ²)": np.var(df[column]),
        "Std Dev (σ)": np.std(df[column]),
        "Max Value": np.max(df[column])
    }

stats = [
    get_validation_stats(df_raw, 'chi_stat', "Raw Data (Before Filter)"),
    get_validation_stats(df_clean, 'chi_stat', "Clean Data (After χ² Filter)"),
    get_validation_stats(df_raw, 'text_length', "Raw Data (Before Filter)"),
    get_validation_stats(df_clean, 'text_length', "Clean Data (After χ² Filter)")
]

results_df = pd.DataFrame(stats)

# Round the numerical columns to 4 decimal places for academic presentation
numerical_cols = ["Mean (μ)", "Variance (σ²)", "Std Dev (σ)", "Max Value"]
results_df[numerical_cols] = results_df[numerical_cols].round(4)

print("\n--- Final Mathematical Validation: Variance Reduction ---")
print(results_df.to_string(index=False))

In [ ]:
# 1. Establish the Data from your Summary Statistics
categories = ['Raw Data\n(Before Filter)', 'Clean Data\n(After χ² Filter)']
chi_variances = [917.2837, 130.4960]
length_variances = [52234.4754, 15189.4360]

# 2. Set up the figure for a side-by-side comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# --- Plot 1: Chi-Squared Variance ---
# Red for Raw (Danger/Noise), Green for Clean (Success)
bars1 = ax1.bar(categories, chi_variances, color=['#e74c3c', '#2ecc71'], edgecolor='black', width=0.6)
ax1.set_title('Variance Drop: Chi-Squared (χ²) Statistic\n(~85% Reduction)', fontsize=14, pad=15)
ax1.set_ylabel('Variance (σ²)', fontsize=12)
ax1.set_ylim(0, max(chi_variances) * 1.2) # Give headroom for labels
ax1.grid(axis='y', linestyle='--', alpha=0.7)

# Add exact value annotations on top of the bars
for bar in bars1:
    yval = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, yval + 15, f'{yval:,.2f}', 
             ha='center', va='bottom', fontsize=12, fontweight='bold')

# --- Plot 2: Text Length Variance ---
# Orange for Raw, Blue for Clean
bars2 = ax2.bar(categories, length_variances, color=['#e67e22', '#3498db'], edgecolor='black', width=0.6)
ax2.set_title('Variance Drop: Text Length\n(~70% Reduction)', fontsize=14, pad=15)
ax2.set_ylabel('Variance (σ²)', fontsize=12)
ax2.set_ylim(0, max(length_variances) * 1.2)
ax2.grid(axis='y', linestyle='--', alpha=0.7)

for bar in bars2:
    yval = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2, yval + 1000, f'{yval:,.2f}', 
             ha='center', va='bottom', fontsize=12, fontweight='bold')

# 3. Add a Master Title and Save the Graphic
plt.suptitle('Mathematical Validation: Variance Reduction Before & After Filtering', 
             fontsize=16, fontweight='bold', y=1.05)

plt.tight_layout()
plt.savefig('variance_reduction_chart.png', bbox_inches='tight', dpi=300)
print("Success! High-resolution chart saved as 'variance_reduction_chart.png'")

In [ ]:
import matplotlib.pyplot as plt

# 1. Set up the figure size for a clean, readable plot
plt.figure(figsize=(10, 6))

# 2. Plot the histogram for the CLEAN Chi-Squared Statistics
# We use a green color (#2ecc71) to represent the mathematically verified, clean data
plt.hist(df_clean['chi_stat'], bins=50, color='#2ecc71', edgecolor='black', alpha=0.8)

# 3. Add titles and labels
plt.title('Distribution of Cleaned Chi-Squared (χ²) Statistics', fontsize=14, fontweight='bold')
plt.xlabel('Chi-Squared (χ²) Distance', fontsize=12)
plt.ylabel('Frequency (Number of Sentences)', fontsize=12)

# 4. Add a vertical line to show where the Mean falls
mean_chi = df_clean['chi_stat'].mean()
plt.axvline(mean_chi, color='red', linestyle='dashed', linewidth=2, label=f'Mean: {mean_chi:.2f}')

# 5. Add a grid and legend for readability
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend()

# 6. Display the plot
plt.tight_layout()
plt.show()

By applying a continuous probability threshold via a chi-squared Goodness-of-Fit test, the anomaly detection pipeline successfully isolated the target language's underlying distribution. The mathematical validity of this approach is proven by the summary statistics: structural variance sigma-squared was reduced by approximately $85\%$, and extreme outliers were successfully truncated. Consequently, the distribution of the cleaned data now closely mirrors a high-quality human corpus

In [ ]:
from scipy.stats import kstest

print("--- Continuous Goodness-of-Fit Tests (Kolmogorov-Smirnov) ---")

# 1. Test the Gamma Distribution (against raw ratios)
# We pass in the parameters you calculated earlier: fit_alpha, fit_loc, fit_scale
gamma_stat, gamma_p = kstest(ratios_array, 'gamma', args=(fit_alpha, fit_loc, fit_scale))

# 2. Test the Beta Distribution (against normalized ratios)
# We pass in the parameters you calculated earlier: a, b, loc, scale
beta_stat, beta_p = kstest(normalized_ratios, 'beta', args=(a, b, loc, scale))

print(f"Gamma K-S Statistic (Distance): {gamma_stat:.4f}")
print(f"Gamma p-value:                  {gamma_p:.4e}\n")

print(f"Beta K-S Statistic (Distance):  {beta_stat:.4f}")
print(f"Beta p-value:                   {beta_p:.4e}")

To determine the optimal continuous probability model for baseline character-to-token ratios, a Kolmogorov-Smirnov goodness-of-fit test was conducted. While the large sample size (N=10,000) naturally resulted in microscopic p-values—reflecting the organic variance inherent in web-scraped text—the test statistics provided a definitive relative performance metric. The Gamma distribution achieved a closer fit, with a maximum K-S distance of 0.0435, outperforming the Beta distribution's distance of 0.0542. Consequently, the asymmetric Gamma model was selected as the mathematically superior filter for the first stage of the anomaly detection pipeline.